In [1]:
# =============================================================================
# GAT-GO — REAL LAB DATA VERSION
# =============================================================================
#
# WHAT CHANGED FROM THE SIMULATED DATA VERSION:
# -----------------------------------------------
# 1. DATA LOADING: Real lab data has NO GO features in the community files.
#    The lab only measures luminescence (Lum), not genomics.
#    To keep a GO-based model, we re-use the GO features from the simulated
#    monoculture_dataset.csv as a "genomic prior" for each strain.
#    Each strain (1–100) maps to the same strain in monoculture_dataset.csv.
#
#    WHY THIS IS VALID: The simulated monoculture_dataset encodes genomic
#    properties (gene orthologues) that are fixed per strain. The real lab
#    uses the same 100 strains, so their GO profiles are the same.
#    The community compositions (which strains appear together) come from
#    the real lab files.
#
# 2. LABEL: Same as GAT-SS — suppression = 1 - 10^Lum (computed from real Lum)
#
# 3. GO FEATURE SELECTION: unchanged (variance filter on monoculture data)
#
# 4. NORMALISATION: both GO features (StandardScaler on monoculture) and
#    the extra ss_lum feature (real Lum from c_p_SS file) are normalised.
#
# 5. NODE FEATURES: 200 GO features (same as simulated) PLUS
#    we optionally concatenate the real ss_lum as feature 201.
#    This is a NEW addition: combining genomic + phenotypic features.
#    The model can then learn which matters more.
#    Set USE_REAL_LUM_FEATURE = True to enable this (recommended).
#
# =============================================================================

import numpy as np
import pandas as pd
import random
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GeoLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# =============================================================================
# SECTION 0 — SETTINGS
# =============================================================================

SEED       = 42
# ── UPDATE THESE PATHS TO YOUR FOLDER ──
DATA_DIR   = r"C:\Users\Pratham Shah\Downloads\Document dataset"
OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads\Document dataset\GAT_GO_Realdata"

N_GO_FEATURES          = 1000
GO_EMBEDDING_DIM       = 64
N_SHAPLEY_PERMUTATIONS = 50
N_EPOCHS               = 300
LR                     = 1e-3
BATCH_SIZE             = 16
TEST_FRAC              = 0.20
N_TOP_GO               = 200

# NEW SETTING: whether to append the real Lum (ss_luminescence) as an extra
# node feature alongside the 200 GO features. Set to True (recommended).
# When True:  node features shape = [n_nodes, 201]  (200 GO + 1 real Lum)
# When False: node features shape = [n_nodes, 200]  (GO only, like simulated)
USE_REAL_LUM_FEATURE = True

STRAIN_COLS = [str(i) for i in range(1, 101)]

os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

TEAL   = "#1a6b6b"
ORANGE = "#d4720c"
NAVY   = "#1c2e4a"
RED    = "#E53935"
GREEN  = "#43A047"
BLUE   = "#1E88E5"
GREY   = "#78909C"

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "white",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "sans-serif",
    "font.size"        : 11,
})

# =============================================================================
# SECTION 1 — LOAD DATA
# =============================================================================

print("=" * 65)
print("  GAT-GO REAL LAB DATA — Loading files")
print("=" * 65)

# ── Load monoculture dataset (for GO features) ─────────────────────────────
# This is still from the SIMULATED data — it contains the GO profiles for
# all 100 strains. The real lab uses the same 100 strains, so their GO
# profiles carry over.
df_mono = pd.read_csv(os.path.join(DATA_DIR, "monoculture_dataset.csv"))

# Extract GO column names from monoculture file
go_cols = [c for c in df_mono.columns if c.startswith("GO_")]
print(f"  GO features in monoculture: {len(go_cols)}")
print(f"  Strains in monoculture    : {len(df_mono)}")

# ── Load single-strain lab measurements (for real Lum feature) ────────────
# Used only if USE_REAL_LUM_FEATURE = True
ss_df = pd.read_csv(os.path.join(DATA_DIR, "c_p_SS_1_10_4.csv"))
ss_df["strain_id"] = ss_df[STRAIN_COLS].idxmax(axis=1).astype(int)
# idxmax(axis=1): find which column has the max value (the "1") per row
strain_lum_map_raw = dict(zip(ss_df["strain_id"], ss_df["Lum"]))
print(f"  Single-strain Lum loaded  : {len(ss_df)} strains")

# ── Load community data (all 5 lab plates) ────────────────────────────────
cc_dfs = []
for plate_num in range(1, 6):
    df    = pd.read_csv(os.path.join(DATA_DIR, f"cc_{plate_num}_1_10_4.csv"))
    df["plate_replicate"] = plate_num
    cc_dfs.append(df)

cc_all = pd.concat(cc_dfs, ignore_index=True)
cc_all["n_strains"]   = cc_all[STRAIN_COLS].sum(axis=1).astype(int)
cc_all["community_id"] = range(len(cc_all))

# Compute suppression label from Lum
# suppression = 1 - 10^Lum  (Lum is already log-normalised)
cc_all["community_suppression"] = np.clip(1.0 - 10.0 ** cc_all["Lum"], 0.0, 1.0)

print(f"  Community rows loaded     : {len(cc_all)}")
print(f"  Community sizes           : {sorted(cc_all['n_strains'].unique())}")

unique_sizes = sorted(cc_all["n_strains"].unique())

# =============================================================================
# SECTION 2 — GO FEATURE SELECTION (UNCHANGED)
# =============================================================================
# This section is identical to the simulated version.
# We still select the top 200 most variable GO features from monoculture data.
# =============================================================================

print(f"\n  GO feature selection ...")

# GO feature matrix from monoculture: [100 strains, 1000 features]
go_from_mono = df_mono[go_cols].values.astype(np.float32)

# np.var(axis=0): variance of each feature across all 100 strains
go_variance    = np.var(go_from_mono, axis=0)

# np.argsort: indices that sort ascending; [::-1]: reverse to descending
top_go_indices = np.argsort(go_variance)[::-1][:N_TOP_GO]
top_go_cols    = [go_cols[i] for i in top_go_indices]

print(f"  Features with zero variance: {(go_variance == 0).sum()}")
print(f"  Keeping top {N_TOP_GO} most variable GO features")

# =============================================================================
# SECTION 3 — NORMALISE FEATURES
# =============================================================================
# THREE things to normalise:
#   A) GO features:    StandardScaler fit on monoculture (100 strains × 200 features)
#   B) Real ss Lum:    StandardScaler fit on single-strain file (100 values)
#
# IMPORTANT: fit scalers BEFORE train/test split to avoid data leakage.
# Here we fit on the monoculture/single-strain files (not community data),
# so there's no leakage regardless of how we split communities.
# =============================================================================

print("\n  Normalising features ...")

# A) GO scaler — UNCHANGED from simulated version
scaler_go = StandardScaler()
# fit_transform: learn mean & std, then apply z-score in one call
go_normalised_mono = scaler_go.fit_transform(
    df_mono[top_go_cols].values.astype(np.float32)
)  # [100, N_TOP_GO]

# B) Real Lum scaler — NEW for real data
# Fit on the 100 single-strain Lum values
scaler_lum = StandardScaler()
lum_values = np.array(list(strain_lum_map_raw.values())).reshape(-1, 1)
# reshape(-1, 1): StandardScaler expects 2D array, -1 = infer size automatically
scaler_lum.fit(lum_values)

# Apply Lum scaler to get normalised Lum per strain
strain_lum_normalised = {}
for strain_id, raw_lum in strain_lum_map_raw.items():
    normed = scaler_lum.transform([[raw_lum]])[0][0]
    strain_lum_normalised[strain_id] = normed

# Build strain_go_map: strain_id (string like "Strain_001") → GO vector [N_TOP_GO]
# Monoculture strain IDs are like "Strain_001", but lab files use integers 1–100.
# We need to map between them.
# Monoculture: Strain_001 = strain 1, Strain_002 = strain 2, etc.
# Extract the integer from the strain_id string.
strain_go_map = {}
for i, strain_id_str in enumerate(df_mono["strain_id"].values):
    # Extract integer: "Strain_001" → 1
    # int(strain_id_str.split("_")[1]) splits on "_" and takes the number part
    try:
        strain_num = int(strain_id_str.split("_")[1])
    except (IndexError, ValueError):
        # Fallback: if format is different, try direct int conversion
        strain_num = int(strain_id_str)
    strain_go_map[strain_num] = go_normalised_mono[i]   # [N_TOP_GO]

print(f"  GO features normalised: mean={go_normalised_mono.mean():.3f}, std={go_normalised_mono.std():.3f}")
print(f"  Strains with GO data: {len(strain_go_map)}")

# Determine final input feature dimension
# If USE_REAL_LUM_FEATURE: 200 GO features + 1 real Lum feature = 201
N_INPUT_FEATURES = N_TOP_GO + (1 if USE_REAL_LUM_FEATURE else 0)
print(f"  Final node feature size: {N_INPUT_FEATURES} "
      f"({'200 GO + 1 Lum' if USE_REAL_LUM_FEATURE else '200 GO only'})")

# =============================================================================
# SECTION 4 — BUILD GRAPHS
# =============================================================================
# WHAT IS DIFFERENT:
#   Simulated: looked up GO features from df_full (community dataset with GO cols).
#   Real lab: look up GO features from strain_go_map (monoculture-derived),
#             AND optionally append the real Lum as extra feature.
#
#   Membership is determined by the one-hot columns in cc_all (same as GAT-SS).
#   Labels come from the Lum-derived suppression (same as GAT-SS).
# =============================================================================

print("\n  Building graphs ...")

graph_list = []
n_missing  = 0

for _, row in cc_all.iterrows():

    # Find which strains are in this community (one-hot → list of strain integers)
    present_strains = [
        int(STRAIN_COLS[i])
        for i, v in enumerate(row[STRAIN_COLS].values)
        if v == 1
    ]
    n_nodes = len(present_strains)

    # Build node feature matrix
    node_feats_list = []
    valid = True

    for s in present_strains:
        if s not in strain_go_map:
            # This strain has no GO data — skip community
            # This shouldn't happen if all 100 strains are in monoculture_dataset
            n_missing += 1
            valid = False
            break

        # Get GO features for this strain: [N_TOP_GO]
        go_feat = strain_go_map[s].copy()  # .copy() to avoid modifying the map

        if USE_REAL_LUM_FEATURE:
            # Get normalised real Lum for this strain
            lum_feat = strain_lum_normalised.get(s, 0.0)
            # np.append: add one value to the end of the array
            # This gives [go_feat..., lum_feat] = [N_TOP_GO + 1] vector
            node_feat = np.append(go_feat, lum_feat)
        else:
            node_feat = go_feat   # [N_TOP_GO] only

        node_feats_list.append(node_feat)

    if not valid:
        continue

    # np.stack(list_of_arrays, axis=0): stack 1D arrays as rows → 2D matrix
    # Result: [n_nodes, N_INPUT_FEATURES]
    node_feats = np.stack(node_feats_list, axis=0)
    x = torch.tensor(node_feats, dtype=torch.float)

    # Fully connected edges — UNCHANGED
    src, dst = [], []
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j:
                src.append(i); dst.append(j)
    edge_index = torch.tensor([src, dst], dtype=torch.long)

    y = torch.tensor([float(row["community_suppression"])], dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, y=y)
    data.community_id    = int(row["community_id"])
    data.community_size  = n_nodes
    data.strain_ids      = present_strains
    data.plate_replicate = int(row["plate_replicate"])
    data.lum_raw         = float(row["Lum"])

    graph_list.append(data)

print(f"  Graphs built  : {len(graph_list)}")
print(f"  Skipped       : {n_missing} (strains with no GO data)")
sample = graph_list[0]
print(f"  Sample graph — nodes: {sample.x.shape[0]}, "
      f"features per node: {sample.x.shape[1]}, "
      f"edges: {sample.edge_index.shape[1]}")

# =============================================================================
# SECTION 5 — TRAIN / TEST SPLIT (stratified, same as GAT-SS)
# =============================================================================

print("\n  Splitting into train / test (stratified by community size) ...")

train_graphs, test_graphs = [], []

for size in unique_sizes:
    size_graphs = [g for g in graph_list if g.community_size == size]
    random.shuffle(size_graphs)
    n_test = max(1, int(len(size_graphs) * TEST_FRAC))
    test_graphs  += size_graphs[:n_test]
    train_graphs += size_graphs[n_test:]

random.shuffle(train_graphs)
random.shuffle(test_graphs)

print(f"  Train: {len(train_graphs)} graphs")
print(f"  Test : {len(test_graphs)} graphs")

train_loader = GeoLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = GeoLoader(test_graphs,  batch_size=BATCH_SIZE, shuffle=False)

# =============================================================================
# SECTION 6 — MODEL
# =============================================================================
# WHAT IS DIFFERENT:
#   in_channels = N_INPUT_FEATURES (200 or 201) instead of N_TOP_GO (200).
#   The go_encoder now optionally handles 201-dim input.
#   All other architecture choices are UNCHANGED.
#
#   If USE_REAL_LUM_FEATURE=True, the model automatically receives the extra
#   Lum channel — no other code changes needed. The encoder handles any
#   input dimension via in_channels.
# =============================================================================

class GATGO(nn.Module):
    """
    Graph Attention Network — Gene Orthologue (GAT-GO), Real Data Version.
    
    Key change vs simulated version:
      in_channels = 200 (GO only) or 201 (GO + real Lum).
      The encoder compresses this to embed_dim=64 before message passing.
      Everything downstream is identical to the simulated version.
    """

    def __init__(self, in_channels, embed_dim, hidden_channels, heads, dropout):
        super(GATGO, self).__init__()

        # GO encoder: compress N_INPUT_FEATURES → embed_dim (64)
        # nn.Sequential chains layers — output of each feeds into the next
        # This now handles 200 or 201 input features transparently
        self.go_encoder = nn.Sequential(
            nn.Linear(in_channels, embed_dim),
            # nn.Linear(in, out): learned matrix multiply + bias
            # Parameters: in_channels * embed_dim + embed_dim (weights + biases)
            nn.ELU(),
            nn.Dropout(p=dropout),
            nn.Linear(embed_dim, embed_dim),
            nn.ELU(),
        )

        # GATConv layers: UNCHANGED from simulated version
        self.conv1 = GATConv(
            embed_dim, hidden_channels,
            heads=heads, concat=True, dropout=dropout, add_self_loops=True
        )
        self.conv2 = GATConv(
            hidden_channels * heads, hidden_channels,
            heads=1, concat=False, dropout=dropout, add_self_loops=True
        )

        self.bn1     = nn.BatchNorm1d(hidden_channels * heads)
        self.bn2     = nn.BatchNorm1d(hidden_channels)
        self.dropout = nn.Dropout(p=dropout)

        # MLP head: UNCHANGED
        self.lin1 = nn.Linear(hidden_channels * 2, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Step 1: encode GO (+ optional Lum) → compact 64-dim embedding
        x = self.go_encoder(x)           # [total_nodes, embed_dim]

        # Step 2: GAT message passing — UNCHANGED
        x = self.conv1(x, edge_index)    # [total_nodes, hidden*heads]
        x = self.bn1(x)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)    # [total_nodes, hidden]
        x = self.bn2(x)
        x = F.elu(x)

        # Step 3: pool — UNCHANGED
        x_mean = global_mean_pool(x, batch)
        x_max  = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1) 

        # Step 4: predict — UNCHANGED
        x = self.dropout(x)
        x = F.elu(self.lin1(x))
        x = torch.sigmoid(self.lin2(x))

        return x.squeeze(-1)

    def get_node_embeddings(self, data):
        """Node embeddings after GO encoding + 2 GAT layers."""
        x, edge_index = data.x, data.edge_index
        x = self.go_encoder(x)
        x = F.elu(self.bn1(self.conv1(x, edge_index)))
        x = F.elu(self.bn2(self.conv2(x, edge_index)))
        return x


# Instantiate: in_channels now depends on USE_REAL_LUM_FEATURE
model = GATGO(
    in_channels     = N_INPUT_FEATURES,  # 200 or 201
    embed_dim       = GO_EMBEDDING_DIM,  # 64
    hidden_channels = 32,
    heads           = 4,
    dropout         = 0.3,
)

print(f"\n  Model architecture:")
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total trainable parameters: {total_params:,}")
print(f"  Input feature size: {N_INPUT_FEATURES}")

# =============================================================================
# SECTION 7 — TRAINING (same as GAT-SS real data version, with best checkpoint)
# =============================================================================

print("\n  Training ...")

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
loss_fn   = nn.HuberLoss(delta=0.2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=40, factor=0.5
)

train_losses, test_losses = [], []
best_test_loss  = float("inf")
best_state_dict = None

for epoch in range(1, N_EPOCHS + 1):

    model.train()
    total_loss, n_batches = 0.0, 0

    for batch in train_loader:
        optimizer.zero_grad()
        pred = model(batch)
        loss = loss_fn(pred, batch.y.squeeze())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        n_batches  += 1

    avg_train_loss = total_loss / n_batches

    model.eval()
    total_test_loss, n_test_batches = 0.0, 0
    with torch.no_grad():
        for batch in test_loader:
            pred = model(batch)
            loss = loss_fn(pred, batch.y.squeeze())
            total_test_loss += loss.item()
            n_test_batches  += 1

    avg_test_loss = total_test_loss / max(n_test_batches, 1)
    train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)
    scheduler.step(avg_test_loss)

    # Save best checkpoint
    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        best_state_dict = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 50 == 0 or epoch == 1:
        print(f"  Epoch {epoch:>3}/{N_EPOCHS} | "
              f"Train: {avg_train_loss:.4f} | "
              f"Test: {avg_test_loss:.4f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.6f}")

print(f"\n  Restoring best model (test loss = {best_test_loss:.4f}) ...")
model.load_state_dict(best_state_dict)

# =============================================================================
# SECTION 8 — EVALUATION
# =============================================================================

print("\n  Evaluating ...")

model.eval()
all_preds, all_true, all_sizes, all_plates = [], [], [], []

with torch.no_grad():
    for data in test_graphs:
        batch_tensor = torch.zeros(data.x.shape[0], dtype=torch.long)
        d    = Data(x=data.x, edge_index=data.edge_index, batch=batch_tensor)
        pred = model(d).item()
        all_preds.append(pred)
        all_true.append(data.y.item())
        all_sizes.append(data.community_size)
        all_plates.append(data.plate_replicate)

all_preds  = np.array(all_preds)
all_true   = np.array(all_true)
all_sizes  = np.array(all_sizes)
all_plates = np.array(all_plates)

r2              = r2_score(all_true, all_preds)
mae             = mean_absolute_error(all_true, all_preds)
mse             = np.mean((all_preds - all_true) ** 2)
spearman_rho, spearman_p = spearmanr(all_true, all_preds)

train_mean   = np.mean([g.y.item() for g in train_graphs])
baseline1_r2 = r2_score(all_true, np.full_like(all_true, train_mean))

print(f"\n  ┌── GAT-GO Real Data Results ({len(test_graphs)} test graphs) ──────┐")
print(f"  │  R²         : {r2:.4f}  (GO {'+ real Lum' if USE_REAL_LUM_FEATURE else 'only'})              │")
print(f"  │  MAE        : {mae:.4f}                                    │")
print(f"  │  MSE        : {mse:.4f}                                    │")
print(f"  │  Spearman ρ : {spearman_rho:.4f}  (p={spearman_p:.3e})             │")
print(f"  ├── Baseline ───────────────────────────────────────────────┤")
print(f"  │  Global mean R² : {baseline1_r2:.4f}                            │")
print(f"  └────────────────────────────────────────────────────────────┘")

print(f"\n  R² per plate replicate:")
for p in sorted(set(all_plates)):
    mask = all_plates == p
    if mask.sum() > 2:
        plate_r2 = r2_score(all_true[mask], all_preds[mask])
        print(f"    Plate {p}: R² = {plate_r2:.4f}  (n={mask.sum()})")

# =============================================================================
# SECTION 9 — GO FEATURE IMPORTANCE (gradient × input, UNCHANGED logic)
# =============================================================================
# The logic is identical to the simulated version.
# The gradient × input attribution works regardless of whether we're on
# real or simulated data — it tells us which GO features the model uses most.
# =============================================================================

print("\n  Computing GO feature importance ...")

model.eval()
# importance accumulates |gradient × input| for each of the N_INPUT_FEATURES
importance_scores = np.zeros(N_INPUT_FEATURES)
n_samples = 0

for data in test_graphs:
    # We need gradients to flow back to data.x
    # Clone and detach first, then set requires_grad=True
    x = data.x.clone().detach().requires_grad_(True)
    # requires_grad_(True): tell PyTorch to track gradients on this tensor
    # Normally input tensors don't get gradients — we have to ask explicitly

    batch_tensor = torch.zeros(x.shape[0], dtype=torch.long)
    # Create a temporary Data object with our gradient-tracking x
    d = Data(x=x, edge_index=data.edge_index, batch=batch_tensor)

    pred = model(d)
    # .sum(): collapse to scalar (needed before backward())
    # If pred has multiple values (batch), backward needs a scalar
    pred.sum().backward()
    # backward(): computes d(pred)/d(x) for all elements of x

    if x.grad is not None:
        # gradient × input: how much does each feature's value × its sensitivity matter?
        # .abs(): take absolute value (sign doesn't matter for importance)
        # .mean(dim=0): average across all nodes in this community → [N_INPUT_FEATURES]
        importance = (x.grad * x).abs().mean(dim=0)
        importance_scores += importance.detach().numpy()
        n_samples += 1

# Average over all test communities
if n_samples > 0:
    importance_scores /= n_samples

# Separate GO importance from Lum importance
go_importance  = importance_scores[:N_TOP_GO]       # first 200 = GO features
lum_importance = importance_scores[N_TOP_GO] if USE_REAL_LUM_FEATURE else None

# Top 20 GO features
top20_idx    = np.argsort(go_importance)[::-1][:20]
top20_labels = [top_go_cols[i] for i in top20_idx]
top20_scores = go_importance[top20_idx]

print(f"  Top 5 most important GO features:")
for i in range(min(5, len(top20_labels))):
    print(f"    {top20_labels[i]}: {top20_scores[i]:.6f}")

if USE_REAL_LUM_FEATURE and lum_importance is not None:
    print(f"  Real Lum feature importance: {lum_importance:.6f}")
    pct = lum_importance / (go_importance.sum() + lum_importance) * 100
    print(f"  Lum contributes {pct:.1f}% of total feature importance")
    # This tells you whether genomics (GO) or phenotype (Lum) matters more!

# =============================================================================
# SECTION 10 — SHAPLEY VALUES (UNCHANGED logic)
# =============================================================================

print("\n  Computing Shapley values ...")

baseline_pred = np.mean([g.y.item() for g in train_graphs])

def compute_shapley_values(model, data, n_permutations=N_SHAPLEY_PERMUTATIONS):
    """
    Permutation-based Shapley values — UNCHANGED from simulated version.
    For GAT-GO: positive Shapley = this strain's GENOME drives suppression.
    """
    n_nodes      = data.x.shape[0]
    shapley_vals = np.zeros(n_nodes)

    for _ in range(n_permutations):
        perm      = np.random.permutation(n_nodes)
        prev_pred = baseline_pred

        for pos, node_idx in enumerate(perm):
            coalition  = perm[:pos + 1].tolist()
            x_masked   = torch.zeros_like(data.x)
            x_masked[coalition] = data.x[coalition]

            masked_data       = Data(x=x_masked, edge_index=data.edge_index)
            masked_data.batch = torch.zeros(n_nodes, dtype=torch.long)

            with torch.no_grad():
                curr_pred = model(masked_data).item()

            marginal = curr_pred - prev_pred
            shapley_vals[node_idx] += marginal
            prev_pred = curr_pred

    shapley_vals /= n_permutations
    return shapley_vals


# Sample communities for Shapley, stratified by size
sample_graphs = []
for size in unique_sizes:
    size_test = [g for g in test_graphs if g.community_size == size]
    n_pick    = max(1, 30 // len(unique_sizes))
    sample_graphs.extend(random.sample(size_test, min(n_pick, len(size_test))))

shapley_results = []
for data in sample_graphs:
    shap_vals = compute_shapley_values(model, data)
    for node_i, shap_val in enumerate(shap_vals):
        strain_num = data.strain_ids[node_i]
        # Recover raw lum for this strain (for scatter plot)
        raw_lum = strain_lum_map_raw.get(strain_num, 0.0)
        shapley_results.append({
            "community_id"   : data.community_id,
            "community_size" : data.community_size,
            "plate_replicate": data.plate_replicate,
            "strain_id"      : strain_num,
            "lum_raw"        : raw_lum,
            "shapley_value"  : shap_val,
            "abs_shapley"    : abs(shap_val),
        })

df_shap = pd.DataFrame(shapley_results)
print(f"  Shapley results: {len(df_shap)} strain-community pairs")
print(f"  Mean |Shapley|  : {df_shap['abs_shapley'].mean():.4f}")

# =============================================================================
# SECTION 11 — FIGURES
# =============================================================================

print("\n  Generating figures ...")

# ── Figure 1: Training curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("GAT-GO Real Data — Training Curves", fontsize=13, fontweight="bold")
epochs_x = range(1, len(train_losses) + 1)
axes[0].semilogy(epochs_x, train_losses, color=TEAL,   lw=2, label="Train")
axes[0].semilogy(epochs_x, test_losses,  color=ORANGE, lw=2, label="Test")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Huber Loss (log scale)")
axes[0].set_title("Loss Curves"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(list(epochs_x)[-100:], train_losses[-100:], color=TEAL,   lw=2, label="Train")
axes[1].plot(list(epochs_x)[-100:], test_losses[-100:],  color=ORANGE, lw=2, label="Test")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Huber Loss")
axes[1].set_title("Final 100 Epochs (zoomed)"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_training_curves_gor.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig1_training_curves_go.png")

# ── Figure 2: Prediction performance ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("GAT-GO Real Data — Prediction Performance", fontsize=13, fontweight="bold")
sc = axes[0].scatter(all_true, all_preds, c=all_sizes, cmap="viridis",
                     s=40, alpha=0.6, edgecolors="none")
plt.colorbar(sc, ax=axes[0], label="Community size (N)")
lims = [min(all_true.min(), all_preds.min())-0.02, max(all_true.max(), all_preds.max())+0.02]
axes[0].plot(lims, lims, "k--", lw=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual suppression"); axes[0].set_ylabel("Predicted suppression")
axes[0].set_title(f"Predicted vs Actual\nR²={r2:.3f}  MAE={mae:.3f}  ρ={spearman_rho:.3f}")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

r2s_by_size = []
for sz in unique_sizes:
    mask = all_sizes == sz
    r2s_by_size.append(r2_score(all_true[mask], all_preds[mask]) if mask.sum() > 2 else float("nan"))
axes[1].bar([str(s) for s in unique_sizes], r2s_by_size, color=TEAL, alpha=0.8, edgecolor="white")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xlabel("Community size"); axes[1].set_ylabel("R²")
axes[1].set_title("R² per Community Size Bin"); axes[1].grid(True, axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_prediction_performance_gor.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig2_prediction_performance_go.png")

# ── Figure 3: GO Feature importance ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("GAT-GO Real Data — Gene Orthologue Feature Importance\n"
             "(gradient × input attribution)", fontsize=13, fontweight="bold")
axes[0].barh(range(20), top20_scores[::-1], color=TEAL, alpha=0.8, edgecolor="white")
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(top20_labels[::-1], fontsize=9)
axes[0].set_xlabel("Importance score")
axes[0].set_title("A) Top 20 Most Important GO Features")
axes[0].grid(True, axis="x", alpha=0.3)

axes[1].hist(go_importance, bins=30, color=TEAL, alpha=0.8, edgecolor="white")
cutoff = np.sort(go_importance)[::-1][N_TOP_GO // 5]
axes[1].axvline(cutoff, color=ORANGE, ls="--", lw=2, label=f"Top {N_TOP_GO // 5} cutoff")
axes[1].set_xlabel("Importance score"); axes[1].set_ylabel("Number of GO features")
axes[1].set_title("B) Distribution of All GO Importances")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig3_go_feature_importance_gor.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig3_go_feature_importance_go.png")

# ── Figure 4: Shapley analysis ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("GAT-GO Real Data — Shapley Values\n(strain-level genomic contribution)",
             fontsize=13, fontweight="bold")
bin_shap = df_shap.groupby("community_size")["abs_shapley"].agg(["mean", "std"]).reset_index()
axes[0].plot(range(len(bin_shap)), bin_shap["mean"], color=TEAL, lw=2.5, marker="o",
             markersize=8, markerfacecolor="white", markeredgewidth=2, markeredgecolor=TEAL)
axes[0].fill_between(range(len(bin_shap)),
                     bin_shap["mean"] - bin_shap["std"],
                     bin_shap["mean"] + bin_shap["std"],
                     alpha=0.15, color=TEAL)
axes[0].set_xticks(range(len(bin_shap)))
axes[0].set_xticklabels(bin_shap["community_size"].astype(str), fontsize=9)
axes[0].set_title("A) Mean |Shapley| vs Community Size\n(genomic roles stabilise with diversity)")
axes[0].set_xlabel("Community size"); axes[0].set_ylabel("Mean absolute Shapley")
axes[0].grid(True, alpha=0.3)

sns.violinplot(data=df_shap, x="community_size", y="shapley_value",
               color=TEAL, alpha=0.6, linewidth=1.2, inner="quartile", ax=axes[1])
axes[1].axhline(0, color=GREY, lw=1.2, ls="--")
axes[1].set_title("B) Shapley Distribution per Size\n(genomic contributions stabilise in larger communities)")
axes[1].set_xlabel("Community size"); axes[1].set_ylabel("Shapley value")
axes[1].tick_params(axis="x", labelsize=8); axes[1].grid(True, axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig4_shapley_analysis_gor.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig4_shapley_analysis_go.png")

# ── Figure 5: Node embeddings + GO correlation heatmap ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("GAT-GO Real Data — Node Embeddings & GO Feature Correlations\n"
             "(what the model learned about each strain's genomic role)",
             fontsize=13, fontweight="bold")

largest_graph = max(test_graphs, key=lambda g: g.community_size)
model.eval()
with torch.no_grad():
    emb = model.get_node_embeddings(largest_graph).numpy()

pca    = PCA(n_components=2)
coords = pca.fit_transform(emb)

# Colour by gene richness: fraction of GO features that are > 0
go_feats    = largest_graph.x[:, :N_TOP_GO].numpy()  # take only GO cols, not Lum
gene_rich   = (go_feats > 0).mean(axis=1)  # fraction of non-zero GO features per strain

sc = axes[0].scatter(coords[:, 0], coords[:, 1], c=gene_rich,
                     cmap="YlOrRd", s=80, edgecolors="white", linewidths=0.5, alpha=0.85)
plt.colorbar(sc, ax=axes[0], label="Gene richness (fraction GO > 0)")
axes[0].set_title(f"A) Node Embeddings (PCA)\nN={largest_graph.community_size} community\n"
                  f"Colour = gene richness")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].grid(True, alpha=0.3)

# Correlation between top 15 GO features across all test community nodes
all_go_feats = np.vstack([g.x[:, :N_TOP_GO].numpy() for g in test_graphs])
top15_labels = [top_go_cols[i].replace("GO_", "") for i in top20_idx[:15]]
top15_mat    = all_go_feats[:, top20_idx[:15]]
corr_mat     = np.corrcoef(top15_mat.T)  # .T transposes so each feature is a row

sns.heatmap(corr_mat, ax=axes[1],
            xticklabels=top15_labels, yticklabels=top15_labels,
            cmap="RdBu_r", vmin=-1, vmax=1, center=0,
            linewidths=0.3, linecolor="#e0e0e0",
            cbar_kws={"label": "Pearson correlation"})
axes[1].set_title("B) Correlation Between Top 15 GO Features\n"
                  "(correlated genes tend to appear together in strains)")
axes[1].tick_params(axis="both", labelsize=8)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig5_embeddings_go_corr_gor.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("  → fig5_embeddings_go_corr_go.png")

# =============================================================================
# SECTION 12 — SAVE
# =============================================================================

import pickle

model_path = os.path.join(OUTPUT_DIR, "gat_go_realdata_model.pt")
torch.save(model.state_dict(), model_path)

shap_path = os.path.join(OUTPUT_DIR, "shapley_values_gat_go_realdata.csv")
df_shap.to_csv(shap_path, index=False)

# Save scalers for later use (e.g. running new communities through the model)
with open(os.path.join(OUTPUT_DIR, "scaler_go.pkl"), "wb") as f:
    pickle.dump(scaler_go, f)
with open(os.path.join(OUTPUT_DIR, "scaler_lum.pkl"), "wb") as f:
    pickle.dump(scaler_lum, f)

# Save the selected GO feature names so you can reproduce the feature selection
with open(os.path.join(OUTPUT_DIR, "top_go_cols.txt"), "w") as f:
    f.write("\n".join(top_go_cols))

print(f"\n  Model saved   : {model_path}")
print(f"  Shapley saved : {shap_path}")
print(f"  Scalers saved : scaler_go.pkl, scaler_lum.pkl")

print("\n" + "=" * 65)
print("  GAT-GO REAL DATA — COMPLETE")
print("=" * 65)
print(f"  Final R²                : {r2:.4f}")
print(f"  Final MAE               : {mae:.4f}")
print(f"  Spearman ρ              : {spearman_rho:.4f}")
print(f"  Feature mode            : {'GO + real Lum (201 features)' if USE_REAL_LUM_FEATURE else 'GO only (200 features)'}")
print(f"  Beats global mean       : {'YES' if r2 > baseline1_r2 else 'NO'}")
print(f"  All outputs in          : {OUTPUT_DIR}")
print("=" * 65)

  GAT-GO REAL LAB DATA — Loading files
  GO features in monoculture: 1000
  Strains in monoculture    : 100
  Single-strain Lum loaded  : 100 strains
  Community rows loaded     : 1750
  Community sizes           : [np.int64(2), np.int64(4), np.int64(6), np.int64(8), np.int64(12), np.int64(16), np.int64(24), np.int64(48)]

  GO feature selection ...
  Features with zero variance: 0
  Keeping top 200 most variable GO features

  Normalising features ...
  GO features normalised: mean=0.000, std=1.000
  Strains with GO data: 100
  Final node feature size: 201 (200 GO + 1 Lum)

  Building graphs ...
  Graphs built  : 1750
  Skipped       : 0 (strains with no GO data)
  Sample graph — nodes: 2, features per node: 201, edges: 2

  Splitting into train / test (stratified by community size) ...
  Train: 1400 graphs
  Test : 350 graphs

  Model architecture:
GATGO(
  (go_encoder): Sequential(
    (0): Linear(in_features=201, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Dropout(